In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------

# Create widgets for configurable parameters
dbutils.widgets.text("num_patients", "327", "Number of Patients")
dbutils.widgets.text("events_per_patient_max", "8", "Max Events Per Patient")
dbutils.widgets.text("hl7_version", "2.5", "HL7 Version")
dbutils.widgets.text("sending_app", "DATABRICKS_SIM", "Sending Application")
dbutils.widgets.text("sending_facility", "DBX_FAC", "Sending Facility")
dbutils.widgets.text("receiving_app", "DOWNSTREAM_SYS", "Receiving Application")
dbutils.widgets.text("receiving_facility", "DST_FAC", "Receiving Facility")
dbutils.widgets.text("catalog_use", "main", "Catalog Name")
dbutils.widgets.text("schema_use", "healthcare", "Schema Name")
dbutils.widgets.text("volume_name", "hl7_synthetic", "Volume Name")
dbutils.widgets.text("relative_path", "adt_fake", "Relative Path")

# Get widget values
num_patients = int(dbutils.widgets.get("num_patients"))
events_per_patient_max = int(dbutils.widgets.get("events_per_patient_max"))
hl7_version = dbutils.widgets.get("hl7_version")
sending_app = dbutils.widgets.get("sending_app")
sending_facility = dbutils.widgets.get("sending_facility")
receiving_app = dbutils.widgets.get("receiving_app")
receiving_facility = dbutils.widgets.get("receiving_facility")
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
volume_name = dbutils.widgets.get("volume_name")
relative_path = dbutils.widgets.get("relative_path")

# Volume path: /Volumes/<catalog>/<schema>/<volume>/<path_to_file>
output_path = f"/Volumes/{catalog_use}/{schema_use}/{volume_name}/{relative_path}"

In [0]:
volume_details = spark.sql(f"""
  SELECT
    v.volume_catalog,
    v.volume_schema,
    v.volume_name,
    v.storage_location,
    e.external_location_name,
    e.url AS external_location_url
  FROM system.information_schema.volumes v
  JOIN system.information_schema.external_locations e
    -- storage_location is a subpath under the external location URL
    ON lower(v.storage_location) LIKE concat(lower(e.url), '%')
  WHERE v.volume_catalog = '{catalog_use}'
    AND v.volume_schema  = '{schema_use}'
    AND v.volume_name    = '{volume_name}'
  ORDER BY length(e.url) DESC   -- pick the most specific matching location
  LIMIT 1
""")

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from datetime import datetime

# ============================================================================
# SEGMENT-SPECIFIC UDFs - Each handles one HL7 segment
# ============================================================================

# --- MSH Segment UDF ---
@udf(returnType=StringType())
def build_msh_segment(msg_datetime, msg_control_id, event_type, sending_app, sending_facility, receiving_app, receiving_facility, hl7_version):
    """Build MSH (Message Header) segment"""
    return f"MSH|^~\\&|{sending_app}|{sending_facility}|{receiving_app}|{receiving_facility}|{msg_datetime}||{event_type}|{msg_control_id}|P|{hl7_version}||"

# --- EVN Segment UDF ---
@udf(returnType=StringType())
def build_evn_segment(event_type, event_datetime):
    """Build EVN (Event Type) segment"""
    return f"EVN|{event_type}|{event_datetime}||"

# --- PID Segment UDF ---
@udf(returnType=StringType())
def build_pid_segment(mrn, ssn, last_name, first_name, middle_initial, birth_date, sex, 
                      race, marital_status, religion, ethnic_group, birth_place,
                      addr_street, addr_city, addr_state, addr_zip, phone_home):
    """Build PID (Patient Identification) segment with Phase 1 & 2 enhancements"""
    return (f"PID|1||{mrn}^^^MRN||{last_name}^{first_name}^{middle_initial}||{birth_date}|{sex}||{race}|" +
            f"{addr_street}^^{addr_city}^{addr_state}^{addr_zip}^USA||{phone_home}|||{marital_status}|" +
            f"{religion}||||{ssn}|||{ethnic_group}|{birth_place}||")

# --- NK1 Segment UDF ---
@udf(returnType=StringType())
def build_nk1_segment(nok_last_name, nok_first_name, nok_relationship, nok_phone):
    """Build NK1 (Next of Kin) segment"""
    return f"NK1|1|{nok_last_name}^{nok_first_name}|{nok_relationship}|{nok_phone}||"

# --- PV1 Segment UDF ---
@udf(returnType=StringType())
def build_pv1_segment(patient_class, patient_location, attending_dr_id, attending_dr_last, attending_dr_first,
                      hospital_service, admit_datetime, discharge_datetime, event_type):
    """Build PV1 (Patient Visit) segment with Phase 1 enhancements"""
    # PV1-45 (discharge datetime) only for A03 events
    discharge_dt = discharge_datetime if event_type == 'A03' else ''
    return (f"PV1|1|{patient_class}|{patient_location}|||{attending_dr_id}^{attending_dr_last}^{attending_dr_first}^^^^MD|" +
            f"|||{hospital_service}|||||||||{attending_dr_id}^{attending_dr_last}^{attending_dr_first}^^^^MD|" +
            f"||||||||||||||||||||||||||||||||{admit_datetime}|{discharge_dt}||")

# --- PV2 Segment UDF ---
@udf(returnType=StringType())
def build_pv2_segment(admit_reason, expected_discharge):
    """Build PV2 (Patient Visit Additional) segment - Phase 2 enhancement"""
    return f"PV2|||{admit_reason}|||||{expected_discharge}|"

# --- DG1 Segment UDF ---
@udf(returnType=StringType())
def build_dg1_segment(diagnosis_code, diagnosis_desc, diagnosis_type, msg_datetime):
    """Build DG1 (Diagnosis) segment - Phase 1 enhancement"""
    return f"DG1|1||{diagnosis_code}^{diagnosis_desc}||{msg_datetime}|{diagnosis_type}||"

# --- AL1 Segment UDF (Conditional) ---
@udf(returnType=StringType())
def build_al1_segments(has_allergies, 
                       allergy1_type, allergy1_code_desc, allergy1_severity, allergy1_reaction,
                       has_allergy2,
                       allergy2_type, allergy2_code_desc, allergy2_severity, allergy2_reaction):
    """Build AL1 (Allergy) segments - Phase 2 enhancement. Returns 0-2 segments."""
    if has_allergies != 'Y':
        return ''  # No allergies
    
    segments = []
    
    # First allergy
    segments.append(f"AL1|1|{allergy1_type}|{allergy1_code_desc}|{allergy1_severity}|{allergy1_reaction}|")
    
    # Second allergy (if exists)
    if has_allergy2 == 'Y':
        segments.append(f"AL1|2|{allergy2_type}|{allergy2_code_desc}|{allergy2_severity}|{allergy2_reaction}|")
    
    return '\r'.join(segments)

# --- GT1 Segment UDF ---
@udf(returnType=StringType())
def build_gt1_segment(last_name, first_name, middle_initial, addr_street, addr_city, addr_state, addr_zip, phone_home, birth_date, sex):
    """Build GT1 (Guarantor) segment"""
    return (f"GT1|1||{last_name}^{first_name}^{middle_initial}||{addr_street}^^{addr_city}^{addr_state}^{addr_zip}^USA|" +
            f"{phone_home}|||{birth_date}|{sex}||01||")

# --- IN1 Segment UDF ---
@udf(returnType=StringType())
def build_in1_segment(ins_plan_id, ins_co_name, ins_co_addr_street, ins_co_addr_city, ins_co_addr_state, ins_co_addr_zip,
                      ins_co_phone, ins_group_num, ins_group_name, ins_policy_num, 
                      last_name, first_name, middle_initial, birth_date, 
                      addr_street, addr_city, addr_state, addr_zip, ins_relationship, ins_effective_date):
    """Build IN1 (Insurance) segment"""
    return (f"IN1|1|{ins_plan_id}|{ins_plan_id}|{ins_co_name}|{ins_co_addr_street}^^{ins_co_addr_city}^{ins_co_addr_state}^{ins_co_addr_zip}^USA|" +
            f"|{ins_co_phone}|{ins_group_num}||||{ins_effective_date}||||{last_name}^{first_name}^{middle_initial}|" +
            f"{ins_relationship}|{birth_date}|{addr_street}^^{addr_city}^{addr_state}^{addr_zip}^USA|||||||||||||||||" +
            f"{ins_policy_num}||||||||||{ins_group_name}||")

# --- IN2 Segment UDF ---
@udf(returnType=StringType())
def build_in2_segment(ssn, ins_expiration_date):
    """Build IN2 (Insurance Additional Information) segment"""
    return f"IN2||{ssn}|||||||||||||||||||||||||||||||||||||||||||||||||||{ins_expiration_date}|"

In [0]:
# -------------------------------------------------------------------
# 1) Generate base patient dimension (synthetic with Faker)
# Mix of existing patients (if table exists) and new synthetic patients
# -------------------------------------------------------------------
from faker import Faker
import random
import string
from datetime import datetime, timedelta

# Check if table exists to reuse patients
table_name = f"{catalog_use}.{schema_use}.adt_hl7_messages"
table_exists = spark.catalog.tableExists(table_name)

if table_exists:
    # Get distinct patients from existing table (30% of requested patients)
    num_existing = int(num_patients * 0.3)
    num_new = num_patients - num_existing
    
    print(f"Table exists - selecting {num_existing} existing patients and generating {num_new} new patients")
    
    # Get distinct patient records from table
    existing_patients = (
        spark.table(table_name)
        .select(
            F.col("patient_id"),
            F.first("visit_id").alias("visit_id")
        )
        .groupBy("patient_id")
        .agg(F.first("visit_id").alias("visit_id"))
        .orderBy(F.rand())
        .limit(num_existing)
    )
    
    # Get full patient data by joining back (take first record per patient)
    # Note: We need to reconstruct patient data from message parsing or use a patient dimension table
    # For now, we'll generate new visit_ids for existing patients but keep their patient_ids
    existing_patient_ids = existing_patients.select("patient_id").distinct()
    
    print(f"Found {existing_patient_ids.count()} existing patients to reuse")
else:
    print(f"Table does not exist - generating {num_patients} new patients")
    num_new = num_patients
    num_existing = 0

# Generate random seed offset (different each run)
seed_offset = random.randint(10000, 20000)

# UDFs to generate realistic patient data using Faker
@F.udf("string")
def generate_first_name(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.first_name()

@F.udf("string")
def generate_last_name(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.last_name()

@F.udf("string")
def generate_dob(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    dob = fake.date_of_birth(minimum_age=18, maximum_age=90)
    return dob.strftime("%Y%m%d")

@F.udf("string")
def generate_patient_id(seed_val, first_name, last_name):
    prefix = (last_name[:2] + first_name[:1]).upper()
    random.seed(seed_val)
    number = str(random.randint(100000, 999999))
    return f"{prefix}{number}"

@F.udf("string")
def generate_visit_id(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    random.seed(seed_val)
    visit_date = fake.date_between(start_date='-30d', end_date='today')
    date_str = visit_date.strftime("%y%m%d")
    chars = string.ascii_uppercase + string.digits
    suffix = ''.join(random.choice(chars) for _ in range(4))
    return f"VIS{date_str}{suffix}"

@F.udf("string")
def generate_ssn(seed_val):
    random.seed(seed_val)
    area = str(random.randint(100, 899))
    group = str(random.randint(10, 99))
    serial = str(random.randint(1000, 9999))
    return f"{area}-{group}-{serial}"

@F.udf("string")
def generate_street_address(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.street_address()

@F.udf("string")
def generate_city(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.city()

@F.udf("string")
def generate_state(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.state_abbr()

@F.udf("string")
def generate_zipcode(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.zipcode()

@F.udf("string")
def generate_phone(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    phone = fake.phone_number()
    digits = ''.join(filter(str.isdigit, phone))[:10]
    if len(digits) == 10:
        return f"{digits[0:3]}-{digits[3:6]}-{digits[6:10]}"
    return phone

@F.udf("string")
def generate_race(seed_val):
    random.seed(seed_val)
    races = [("2106-3", 60), ("2054-5", 20), ("2028-9", 10), ("2076-8", 5), ("1002-5", 5)]
    roll = random.randint(1, 100)
    cumulative = 0
    for code, weight in races:
        cumulative += weight
        if roll <= cumulative:
            return code
    return "2106-3"

@F.udf("string")
def generate_marital_status(seed_val):
    random.seed(seed_val)
    statuses = ["M", "S", "D", "W", "A"]
    weights = [45, 30, 15, 7, 3]
    roll = random.randint(1, 100)
    cumulative = 0
    for status, weight in zip(statuses, weights):
        cumulative += weight
        if roll <= cumulative:
            return status
    return "M"

@F.udf("string")
def generate_religion(seed_val):
    random.seed(seed_val)
    religions = ["CAT", "BAP", "PRO", "JEW", "MOS", "OTH", "NON"]
    weights = [25, 15, 20, 5, 3, 12, 20]
    roll = random.randint(1, 100)
    cumulative = 0
    for religion, weight in zip(religions, weights):
        cumulative += weight
        if roll <= cumulative:
            return religion
    return "OTH"

@F.udf("string")
def generate_ethnic_group(seed_val):
    random.seed(seed_val)
    return "H" if random.randint(1, 100) <= 18 else "N"

@F.udf("string")
def generate_birth_place(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return f"{fake.city()}, {fake.state_abbr()}"

@F.udf("string")
def generate_diagnosis_code(seed_val):
    random.seed(seed_val)
    diagnoses = ["E11.9", "I10", "J44.9", "I25.10", "N18.3", "J18.9", "I50.9", "K21.9", "M25.50", "E78.5", "F32.9", "M54.5", "I63.9", "J96.00", "N39.0"]
    return random.choice(diagnoses)

@F.udf("string")
def generate_diagnosis_desc(diagnosis_code):
    desc_map = {
        "E11.9": "Type 2 diabetes mellitus without complications",
        "I10": "Essential hypertension",
        "J44.9": "COPD, unspecified",
        "I25.10": "Atherosclerotic heart disease",
        "N18.3": "Chronic kidney disease, stage 3",
        "J18.9": "Pneumonia, unspecified",
        "I50.9": "Heart failure, unspecified",
        "K21.9": "Gastro-esophageal reflux disease",
        "M25.50": "Pain in unspecified joint",
        "E78.5": "Hyperlipidemia, unspecified",
        "F32.9": "Major depressive disorder",
        "M54.5": "Low back pain",
        "I63.9": "Cerebral infarction, unspecified",
        "J96.00": "Acute respiratory failure",
        "N39.0": "Urinary tract infection",
    }
    return desc_map.get(diagnosis_code, "Unspecified diagnosis")

@F.udf("string")
def generate_diagnosis_type(seed_val):
    random.seed(seed_val)
    return random.choice(["A", "W", "F"])

@F.udf("string")
def generate_hospital_service(seed_val):
    random.seed(seed_val)
    services = ["MED", "SUR", "OBS", "CAR", "NEU", "ONC", "ORT", "PSY", "PED"]
    weights = [30, 20, 15, 10, 8, 7, 5, 3, 2]
    roll = random.randint(1, 100)
    cumulative = 0
    for service, weight in zip(services, weights):
        cumulative += weight
        if roll <= cumulative:
            return service
    return "MED"

@F.udf("string")
def generate_has_allergies(seed_val):
    random.seed(seed_val)
    return "Y" if random.randint(1, 100) <= 30 else "N"

@F.udf("string")
def generate_allergy_type(seed_val):
    random.seed(seed_val)
    return random.choice(["DA", "FA", "MA", "EA"])

@F.udf("string")
def generate_allergy_code_desc(seed_val, allergy_type):
    random.seed(seed_val)
    drug_allergies = ["1191^Aspirin", "7980^Penicillin", "203^Ibuprofen", "1605^Codeine", "1272^Morphine"]
    food_allergies = ["F01^Peanuts", "F02^Shellfish", "F03^Eggs", "F04^Milk", "F05^Soy"]
    misc_allergies = ["M01^Latex", "M02^Contrast dye", "M03^Adhesive tape"]
    env_allergies = ["E01^Pollen", "E02^Dust mites", "E03^Pet dander"]
    
    if allergy_type == "DA":
        return random.choice(drug_allergies)
    elif allergy_type == "FA":
        return random.choice(food_allergies)
    elif allergy_type == "MA":
        return random.choice(misc_allergies)
    else:
        return random.choice(env_allergies)

@F.udf("string")
def generate_allergy_severity(seed_val):
    random.seed(seed_val)
    return random.choice(["MI", "MO", "SV"])

@F.udf("string")
def generate_allergy_reaction(seed_val):
    random.seed(seed_val)
    return random.choice(["Rash", "Hives", "Itching", "Swelling", "Anaphylaxis", "Nausea", "Vomiting"])

@F.udf("string")
def generate_admit_reason(seed_val):
    random.seed(seed_val)
    return random.choice(["Chest pain", "Shortness of breath", "Abdominal pain", "Fever", "Trauma", "Scheduled surgery", "Weakness", "Altered mental status", "Fall", "Syncope"])

@F.udf("string")
def generate_insurance_company(seed_val):
    random.seed(seed_val)
    return random.choice(["Blue Cross", "Aetna", "UnitedHealth", "Cigna", "Humana", "Kaiser", "Medicare", "Medicaid"])

@F.udf("string")
def generate_insurance_id(seed_val):
    random.seed(seed_val)
    return ''.join([random.choice(string.ascii_uppercase + string.digits) for _ in range(12)])

@F.udf("string")
def generate_insurance_group(seed_val):
    random.seed(seed_val)
    return f"GRP{random.randint(10000, 99999)}"

@F.udf("string")
def generate_insurance_plan_id(seed_val):
    random.seed(seed_val)
    return f"PLAN{random.randint(1000, 9999)}"

@F.udf("string")
def generate_insurance_plan_type(seed_val):
    random.seed(seed_val)
    return random.choice(["HMO", "PPO", "EPO", "POS"])

@F.udf("string")
def generate_insurance_effective_date(seed_val):
    random.seed(seed_val)
    fake = Faker()
    Faker.seed(seed_val)
    date = fake.date_between(start_date='-2y', end_date='today')
    return date.strftime("%Y%m%d")

@F.udf("string")
def generate_insurance_expiration_date(seed_val):
    random.seed(seed_val)
    fake = Faker()
    Faker.seed(seed_val)
    date = fake.date_between(start_date='today', end_date='+1y')
    return date.strftime("%Y%m%d")

# Generate NEW patients dimension
new_patients_df = (
    spark.range(num_new)
    .withColumn("patient_index", F.col("id"))
    .withColumn("seed", (F.col("id") + seed_offset).cast("int"))
    .withColumn("first_name", generate_first_name(F.col("seed")))
    .withColumn("last_name", generate_last_name(F.col("seed") + 1))
    .withColumn("dob", generate_dob(F.col("seed") + 2))
    .withColumn("patient_id", generate_patient_id(F.col("seed"), F.col("first_name"), F.col("last_name")))
    .withColumn("visit_id", generate_visit_id(F.col("seed") + 3))
    .withColumn("sex", F.when(F.col("seed") % 2 == 0, "M").otherwise("F"))
    .withColumn("ssn", generate_ssn(F.col("seed") + 4))
    .withColumn("street_address", generate_street_address(F.col("seed") + 5))
    .withColumn("city", generate_city(F.col("seed") + 6))
    .withColumn("state", generate_state(F.col("seed") + 7))
    .withColumn("zipcode", generate_zipcode(F.col("seed") + 8))
    .withColumn("country", F.lit("USA"))
    .withColumn("phone", generate_phone(F.col("seed") + 9))
    .withColumn("race", generate_race(F.col("seed") + 10))
    .withColumn("marital_status", generate_marital_status(F.col("seed") + 11))
    .withColumn("religion", generate_religion(F.col("seed") + 12))
    .withColumn("ethnic_group", generate_ethnic_group(F.col("seed") + 13))
    .withColumn("birth_place", generate_birth_place(F.col("seed") + 14))
    .withColumn("nok_first_name", generate_first_name(F.col("seed") + 20))
    .withColumn("nok_last_name", generate_last_name(F.col("seed") + 21))
    .withColumn("nok_phone", generate_phone(F.col("seed") + 22))
    .withColumn("nok_relationship", F.when(F.col("seed") % 3 == 0, "SPO").when(F.col("seed") % 3 == 1, "CHD").otherwise("PAR"))
    .withColumn("diagnosis_code", generate_diagnosis_code(F.col("seed") + 30))
    .withColumn("diagnosis_desc", generate_diagnosis_desc(F.col("diagnosis_code")))
    .withColumn("diagnosis_type", generate_diagnosis_type(F.col("seed") + 31))
    .withColumn("hospital_service", generate_hospital_service(F.col("seed") + 32))
    .withColumn("has_allergies", generate_has_allergies(F.col("seed") + 40))
    .withColumn("allergy1_type", F.when(F.col("has_allergies") == "Y", generate_allergy_type(F.col("seed") + 41)))
    .withColumn("allergy1_code_desc", F.when(F.col("has_allergies") == "Y", generate_allergy_code_desc(F.col("seed") + 42, F.col("allergy1_type"))))
    .withColumn("allergy1_severity", F.when(F.col("has_allergies") == "Y", generate_allergy_severity(F.col("seed") + 43)))
    .withColumn("allergy1_reaction", F.when(F.col("has_allergies") == "Y", generate_allergy_reaction(F.col("seed") + 44)))
    .withColumn("has_allergy2", F.when((F.col("has_allergies") == "Y") & (F.col("seed") % 2 == 0), "Y").otherwise("N"))
    .withColumn("allergy2_type", F.when(F.col("has_allergy2") == "Y", generate_allergy_type(F.col("seed") + 45)))
    .withColumn("allergy2_code_desc", F.when(F.col("has_allergy2") == "Y", generate_allergy_code_desc(F.col("seed") + 46, F.col("allergy2_type"))))
    .withColumn("allergy2_severity", F.when(F.col("has_allergy2") == "Y", generate_allergy_severity(F.col("seed") + 47)))
    .withColumn("allergy2_reaction", F.when(F.col("has_allergy2") == "Y", generate_allergy_reaction(F.col("seed") + 48)))
    .withColumn("admit_reason", generate_admit_reason(F.col("seed") + 50))
    .withColumn("insurance_company", generate_insurance_company(F.col("seed") + 60))
    .withColumn("insurance_company_id", F.concat(F.lit("INS"), F.lpad((F.col("seed") % 100).cast("string"), 3, "0")))
    .withColumn("insurance_id", generate_insurance_id(F.col("seed") + 61))
    .withColumn("insurance_group", generate_insurance_group(F.col("seed") + 62))
    .withColumn("insurance_plan_id", generate_insurance_plan_id(F.col("seed") + 63))
    .withColumn("insurance_plan_type", generate_insurance_plan_type(F.col("seed") + 64))
    .withColumn("insurance_effective_date", generate_insurance_effective_date(F.col("seed") + 65))
    .withColumn("insurance_expiration_date", generate_insurance_expiration_date(F.col("seed") + 66))
    .drop("id", "seed")
)

# If table exists, also generate data for EXISTING patients with new visit_ids
if table_exists and num_existing > 0:
    existing_patients_df = (
        existing_patient_ids
        .withColumn("patient_index", F.monotonically_increasing_id())
        .withColumn("seed", (F.monotonically_increasing_id() + seed_offset + 50000).cast("int"))
        .withColumn("first_name", generate_first_name(F.col("seed")))
        .withColumn("last_name", generate_last_name(F.col("seed") + 1))
        .withColumn("dob", generate_dob(F.col("seed") + 2))
        .withColumn("visit_id", generate_visit_id(F.col("seed") + 3))  # New visit for existing patient
        .withColumn("sex", F.when(F.col("seed") % 2 == 0, "M").otherwise("F"))
        .withColumn("ssn", generate_ssn(F.col("seed") + 4))
        .withColumn("street_address", generate_street_address(F.col("seed") + 5))
        .withColumn("city", generate_city(F.col("seed") + 6))
        .withColumn("state", generate_state(F.col("seed") + 7))
        .withColumn("zipcode", generate_zipcode(F.col("seed") + 8))
        .withColumn("country", F.lit("USA"))
        .withColumn("phone", generate_phone(F.col("seed") + 9))
        .withColumn("race", generate_race(F.col("seed") + 10))
        .withColumn("marital_status", generate_marital_status(F.col("seed") + 11))
        .withColumn("religion", generate_religion(F.col("seed") + 12))
        .withColumn("ethnic_group", generate_ethnic_group(F.col("seed") + 13))
        .withColumn("birth_place", generate_birth_place(F.col("seed") + 14))
        .withColumn("nok_first_name", generate_first_name(F.col("seed") + 20))
        .withColumn("nok_last_name", generate_last_name(F.col("seed") + 21))
        .withColumn("nok_phone", generate_phone(F.col("seed") + 22))
        .withColumn("nok_relationship", F.when(F.col("seed") % 3 == 0, "SPO").when(F.col("seed") % 3 == 1, "CHD").otherwise("PAR"))
        .withColumn("diagnosis_code", generate_diagnosis_code(F.col("seed") + 30))
        .withColumn("diagnosis_desc", generate_diagnosis_desc(F.col("diagnosis_code")))
        .withColumn("diagnosis_type", generate_diagnosis_type(F.col("seed") + 31))
        .withColumn("hospital_service", generate_hospital_service(F.col("seed") + 32))
        .withColumn("has_allergies", generate_has_allergies(F.col("seed") + 40))
        .withColumn("allergy1_type", F.when(F.col("has_allergies") == "Y", generate_allergy_type(F.col("seed") + 41)))
        .withColumn("allergy1_code_desc", F.when(F.col("has_allergies") == "Y", generate_allergy_code_desc(F.col("seed") + 42, F.col("allergy1_type"))))
        .withColumn("allergy1_severity", F.when(F.col("has_allergies") == "Y", generate_allergy_severity(F.col("seed") + 43)))
        .withColumn("allergy1_reaction", F.when(F.col("has_allergies") == "Y", generate_allergy_reaction(F.col("seed") + 44)))
        .withColumn("has_allergy2", F.when((F.col("has_allergies") == "Y") & (F.col("seed") % 2 == 0), "Y").otherwise("N"))
        .withColumn("allergy2_type", F.when(F.col("has_allergy2") == "Y", generate_allergy_type(F.col("seed") + 45)))
        .withColumn("allergy2_code_desc", F.when(F.col("has_allergy2") == "Y", generate_allergy_code_desc(F.col("seed") + 46, F.col("allergy2_type"))))
        .withColumn("allergy2_severity", F.when(F.col("has_allergy2") == "Y", generate_allergy_severity(F.col("seed") + 47)))
        .withColumn("allergy2_reaction", F.when(F.col("has_allergy2") == "Y", generate_allergy_reaction(F.col("seed") + 48)))
        .withColumn("admit_reason", generate_admit_reason(F.col("seed") + 50))
        .withColumn("insurance_company", generate_insurance_company(F.col("seed") + 60))
        .withColumn("insurance_company_id", F.concat(F.lit("INS"), F.lpad((F.col("seed") % 100).cast("string"), 3, "0")))
        .withColumn("insurance_id", generate_insurance_id(F.col("seed") + 61))
        .withColumn("insurance_group", generate_insurance_group(F.col("seed") + 62))
        .withColumn("insurance_plan_id", generate_insurance_plan_id(F.col("seed") + 63))
        .withColumn("insurance_plan_type", generate_insurance_plan_type(F.col("seed") + 64))
        .withColumn("insurance_effective_date", generate_insurance_effective_date(F.col("seed") + 65))
        .withColumn("insurance_expiration_date", generate_insurance_expiration_date(F.col("seed") + 66))
        .drop("seed")
    )
    
    # Union existing and new patients
    patients_df = existing_patients_df.unionByName(new_patients_df)
    print(f"Combined {num_existing} existing + {num_new} new = {patients_df.count()} total patients")
else:
    patients_df = new_patients_df
    print(f"Generated {patients_df.count()} new patients")

In [0]:
# -------------------------------------------------------------------
# 2) Event plan per patient with realistic event types
# -------------------------------------------------------------------
event_schema = T.ArrayType(
    T.StructType([
        T.StructField("event_order", T.IntegerType(), False),
        T.StructField("event_type", T.StringType(), False)
    ])
)

@F.udf(event_schema)
def build_event_plan(random_int):
    events = []
    order = 0
    
    journey_type = random_int % 10
    
    if journey_type == 0:
        events.append({"event_order": order, "event_type": "A04"})
        order += 1
        events.append({"event_order": order, "event_type": "A06"})
        order += 1
        for _ in range((random_int // 10) % 3):
            events.append({"event_order": order, "event_type": "A08"})
            order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 1:
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        for _ in range((random_int // 20) % 2 + 1):
            events.append({"event_order": order, "event_type": "A02"})
            order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 2:
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A11"})
        
    elif journey_type == 3:
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A02"})
        order += 1
        events.append({"event_order": order, "event_type": "A12"})
        order += 1
        events.append({"event_order": order, "event_type": "A02"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 4:
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        order += 1
        events.append({"event_order": order, "event_type": "A13"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 5:
        events.append({"event_order": order, "event_type": "A04"})
        order += 1
        if (random_int // 50) % 2 == 0:
            events.append({"event_order": order, "event_type": "A08"})
            
    elif journey_type == 6:
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A07"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    else:
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        
        n_updates = (random_int % 4)
        for _ in range(n_updates):
            events.append({"event_order": order, "event_type": "A08"})
            order += 1
            
        n_transfers = (random_int // 4) % 3
        for _ in range(n_transfers):
            events.append({"event_order": order, "event_type": "A02"})
            order += 1
            
        if (random_int % 5 != 0):
            events.append({"event_order": order, "event_type": "A03"})

    return events

events_planned = (
    patients_df
    .withColumn("rand", (F.rand() * 1000).cast("int"))
    .withColumn("events", build_event_plan(F.col("rand")))
    .drop("rand")
)

events_exploded = (
    events_planned
    .withColumn("event", F.explode("events"))
    .withColumn("event_order", F.col("event.event_order"))
    .withColumn("event_type", F.col("event.event_type"))
    .drop("event", "events")
)

In [0]:
# -------------------------------------------------------------------
# 3) Add timing and visit context
# -------------------------------------------------------------------
from pyspark.sql import Window

base_ts = F.to_timestamp(F.lit("2026-01-01 00:00:00"), "yyyy-MM-dd HH:mm:ss")

events_timed = (
    events_exploded
    .withColumn("hours_offset", (F.col("event_order") * 2 + (F.hash(F.col("patient_id")) % 24)).cast("int"))
    .withColumn(
        "event_datetime",
        F.date_format(
            base_ts + F.expr("MAKE_INTERVAL(0, 0, 0, 0, hours_offset, 0, 0)"),
            "yyyyMMddHHmmss"
        )
    )
    .withColumn("msg_control_id", F.concat(F.lit("MSG"), F.monotonically_increasing_id().cast("string")))
    .withColumn("msg_datetime", F.col("event_datetime"))
    .withColumn(
        "patient_class",
        F.when(F.col("event_type") == "A04", "E")
         .when(F.col("event_type") == "A06", "I")
         .when(F.col("event_type") == "A07", "O")
         .when(F.col("event_type").isin("A01", "A02", "A03"), "I")
         .when(F.col("event_type").isin("A11", "A12", "A13"), "I")
         .otherwise("O")
    )
    .withColumn(
        "patient_location",
        F.concat(
            F.when(F.col("event_type") == "A04", F.lit("ER"))
             .otherwise(F.lit("WARD")),
            F.lpad((F.hash(F.col("patient_id")) % 10).cast("int").cast("string"), 2, "0"),
            F.lit("-ROOM"),
            F.lpad((F.hash(F.col("visit_id")) % 20).cast("int").cast("string"), 2, "0")
        )
    )
    .withColumn("attending_dr_id", F.concat(F.lit("DR"), F.lpad((F.hash(F.col("patient_id")) % 50).cast("int").cast("string"), 3, "0")))
    .withColumn("attending_dr_last", F.concat(F.lit("DOC"), F.lpad((F.hash(F.col("patient_id")) % 50).cast("int").cast("string"), 3, "0")))
    .withColumn("attending_dr_first", F.lit("ATTENDING"))
    .withColumn(
        "discharge_datetime",
        F.when(F.col("event_type") == "A03", F.col("event_datetime")).otherwise("")
    )
    .withColumn(
        "admit_datetime",
        F.first(
            F.when(F.col("event_type").isin("A01", "A04"), F.col("event_datetime")).otherwise(None)
        ).over(Window.partitionBy("patient_id").orderBy("event_order").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
    )
    .withColumn(
        "expected_discharge_ts",
        F.when(
            F.col("admit_datetime").isNotNull(),
            F.to_timestamp(F.col("admit_datetime"), "yyyyMMddHHmmss") + F.expr("MAKE_INTERVAL(0, 0, 0, 3 + (abs(hash(patient_id)) % 5), 0, 0, 0)")
        )
    )
    .withColumn(
        "expected_discharge",
        F.when(F.col("expected_discharge_ts").isNotNull(), F.date_format(F.col("expected_discharge_ts"), "yyyyMMddHHmmss")).otherwise("")
    )
    .drop("hours_offset", "expected_discharge_ts")
)

In [0]:
from pyspark.sql import functions as F

hl7_config = {
    'sending_app': 'SyntheticEHR',
    'sending_facility': 'TestHospital',
    'receiving_app': 'DataPlatform',
    'receiving_facility': 'Analytics',
    'hl7_version': '2.5'
}

with_segments = events_timed \
    .withColumn('seg_msh', build_msh_segment(
        F.col('msg_datetime'), F.col('msg_control_id'), F.col('event_type'),
        F.lit(hl7_config['sending_app']), F.lit(hl7_config['sending_facility']),
        F.lit(hl7_config['receiving_app']), F.lit(hl7_config['receiving_facility']),
        F.lit(hl7_config['hl7_version'])
    )) \
    .withColumn('seg_evn', build_evn_segment(
        F.col('event_type'), F.col('event_datetime')
    )) \
    .withColumn('seg_pid', build_pid_segment(
        F.col('patient_id'),
        F.col('ssn'), 
        F.col('last_name'), 
        F.col('first_name'), 
        F.lit(''),
        F.col('dob'),
        F.col('sex'), 
        F.col('race'), 
        F.col('marital_status'), 
        F.col('religion'), 
        F.col('ethnic_group'), 
        F.col('birth_place'),
        F.col('street_address'),
        F.col('city'),
        F.col('state'),
        F.col('zipcode'),
        F.col('phone')
    )) \
    .withColumn('seg_nk1', build_nk1_segment(
        F.col('nok_last_name'), F.col('nok_first_name'), F.col('nok_relationship'), F.col('nok_phone')
    )) \
    .withColumn('seg_pv1', build_pv1_segment(
        F.col('patient_class'), 
        F.col('patient_location'),
        F.col('attending_dr_id'), 
        F.col('attending_dr_last'), 
        F.col('attending_dr_first'),
        F.col('hospital_service'), 
        F.col('admit_datetime'), 
        F.col('discharge_datetime'), 
        F.col('event_type')
    )) \
    .withColumn('seg_pv2', build_pv2_segment(
        F.col('admit_reason'), F.col('expected_discharge')
    )) \
    .withColumn('seg_dg1', build_dg1_segment(
        F.col('diagnosis_code'), F.col('diagnosis_desc'), F.col('diagnosis_type'), F.col('msg_datetime')
    )) \
    .withColumn('seg_al1', build_al1_segments(
        F.col('has_allergies'),
        F.col('allergy1_type'), F.col('allergy1_code_desc'), F.col('allergy1_severity'), F.col('allergy1_reaction'),
        F.col('has_allergy2'),
        F.col('allergy2_type'), F.col('allergy2_code_desc'), F.col('allergy2_severity'), F.col('allergy2_reaction')
    )) \
    .withColumn('seg_gt1', build_gt1_segment(
        F.col('last_name'), 
        F.col('first_name'), 
        F.lit(''),
        F.col('street_address'),
        F.col('city'),
        F.col('state'),
        F.col('zipcode'),
        F.col('phone'),
        F.col('dob'),
        F.col('sex')
    )) \
    .withColumn('seg_in1', build_in1_segment(
        F.col('insurance_plan_id'), 
        F.col('insurance_company'),
        F.lit(''),
        F.lit(''),
        F.lit(''),
        F.lit(''),
        F.lit(''),
        F.col('insurance_group'),
        F.lit(''),
        F.col('insurance_id'),
        F.col('last_name'), 
        F.col('first_name'), 
        F.lit(''),
        F.col('dob'),
        F.col('street_address'),
        F.col('city'),
        F.col('state'),
        F.col('zipcode'),
        F.lit('18'),
        F.col('insurance_effective_date')
    )) \
    .withColumn('seg_in2', build_in2_segment(
        F.col('ssn'), F.col('insurance_expiration_date')
    ))

hl7_messages = with_segments.withColumn(
    'hl7_message',
    F.concat_ws(
        '\r',
        F.col('seg_msh'),
        F.col('seg_evn'),
        F.col('seg_pid'),
        F.col('seg_nk1'),
        F.col('seg_pv1'),
        F.col('seg_pv2'),
        F.col('seg_dg1'),
        F.when(F.col('seg_al1') != '', F.col('seg_al1')).otherwise(F.lit(None)),
        F.col('seg_gt1'),
        F.col('seg_in1'),
        F.col('seg_in2')
    )
)

In [0]:
# -------------------------------------------------------------------
# Write HL7 messages to Unity Catalog table and individual files
# -------------------------------------------------------------------
import time
import uuid

# Generate batch ID for this run
batch_id = str(uuid.uuid4())
print(f"Batch ID for this run: {batch_id}")

table_name = f"{catalog_use}.{schema_use}.adt_hl7_messages"
hl7_output_path = f"{output_path}/hl7_files"
txt_output_path = f"{output_path}/txt_files"

print(f"Preparing to write messages to {table_name}...")

# Stage 1: Create table with proper DDL if it doesn't exist
table_exists = spark.catalog.tableExists(table_name)

if not table_exists:
    print(f"Creating table {table_name} with proper schema and Delta properties...")
    
    create_table_ddl = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        message_pk STRING GENERATED ALWAYS AS (md5(concat_ws('|', patient_id, visit_id, event_type, event_datetime))),
        patient_id STRING COMMENT 'Unique patient identifier',
        visit_id STRING COMMENT 'Unique visit/encounter identifier',
        event_type STRING COMMENT 'HL7 ADT event type (A01-A13)',
        event_datetime STRING COMMENT 'Event date and time (yyyyMMddHHmmss format)',
        event_order INT COMMENT 'Sequential order of events within patient journey',
        hl7_message STRING COMMENT 'Complete HL7 ADT message with all segments',
        batch_id STRING COMMENT 'Unique identifier for each batch/run that wrote this record',
        created_at TIMESTAMP COMMENT 'Timestamp when record was inserted into table',
        PRIMARY KEY(message_pk)
    )
    USING DELTA
    COMMENT 'HL7 ADT messages for synthetic patient encounters. Each record represents a single ADT event with unique composite key (patient_id, visit_id, event_type, event_datetime).'
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true',
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '7',
        'delta.dataSkippingNumIndexedCols' = '10'
    )
    """
    
    spark.sql(create_table_ddl)
    print(f"Created table {table_name}")
else:
    print(f"Table {table_name} exists - will merge new data")

# Stage 2: Prepare data with PK hash and batch_id
messages_for_table = (
    hl7_messages
    .select(
        F.col("patient_id"),
        F.col("visit_id"),
        F.col("event_type"),
        F.col("event_datetime"),
        F.col("event_order"),
        F.col("hl7_message"),
        F.lit(batch_id).alias("batch_id"),
        F.current_timestamp().alias("created_at")
    )
    .withColumn(
        "message_pk",
        F.md5(F.concat_ws("|", F.col("patient_id"), F.col("visit_id"), F.col("event_type"), F.col("event_datetime")))
    )
    .coalesce(4)
)

row_count = messages_for_table.count()
print(f"Prepared {row_count} messages for table write")

# Stage 3: Write to Delta table with merge for uniqueness
if table_exists:
    messages_for_table.createOrReplaceTempView("new_messages")
    
    merge_query = f"""
    MERGE INTO {table_name} AS target
    USING new_messages AS source
    ON target.message_pk = source.message_pk
    WHEN MATCHED THEN
      UPDATE SET
        target.event_order = source.event_order,
        target.hl7_message = source.hl7_message,
        target.batch_id = source.batch_id,
        target.created_at = source.created_at
    WHEN NOT MATCHED THEN
      INSERT (message_pk, patient_id, visit_id, event_type, event_datetime, event_order, hl7_message, batch_id, created_at)
      VALUES (source.message_pk, source.patient_id, source.visit_id, source.event_type, source.event_datetime,
              source.event_order, source.hl7_message, source.batch_id, source.created_at)
    """
    
    spark.sql(merge_query)
    print(f"Merged data into {table_name}")
else:
    # First insert
    messages_for_table.write.format("delta").mode("append").saveAsTable(table_name)
    print(f"Inserted initial data into {table_name}")

# Optimize table (liquid clustering will auto-optimize over time)
spark.sql(f"OPTIMIZE {table_name}")

# Get final row count
final_count = spark.table(table_name).count()
print(f"Table now contains {final_count} total messages")

# Stage 4: Prepare data for file writes (only this run's data using batch_id)
messages_for_files = (
    spark.table(table_name)
    .filter(F.col("batch_id") == batch_id)
    .withColumn(
        "filename",
        F.concat(
            F.col("patient_id"),
            F.lit("_"),
            F.col("visit_id"),
            F.lit("_"),
            F.col("event_type"),
            F.lit("_"),
            F.col("event_datetime"),
            F.lit(".hl7")
        )
    )
    .select("filename", "hl7_message")
)

messages_list = messages_for_files.collect()
print(f"Writing {len(messages_list)} files from batch {batch_id}...")

# Stage 5: Write individual .hl7 files
dbutils.fs.mkdirs(hl7_output_path)

start_time = time.time()
for i, row in enumerate(messages_list, 1):
    dbutils.fs.put(f"{hl7_output_path}/{row['filename']}", row["hl7_message"], overwrite=True)
    if i % 100 == 0:
        print(f"Progress: {i}/{len(messages_list)} files written")

write_time = time.time() - start_time

# Stage 6: Create 10 sample .txt files
dbutils.fs.mkdirs(txt_output_path)
for row in messages_list[:10]:
    filename = row["filename"].replace(".hl7", ".txt")
    dbutils.fs.put(f"{txt_output_path}/{filename}", row["hl7_message"], overwrite=True)

# Final summary
storage_location_str = volume_details.select('storage_location').first()[0] if volume_details.count() > 0 else 'N/A'
print(f"\n{'='*70}")
print(f"SUCCESS: Wrote to table and {len(messages_list)} files")
print(f"  Batch ID: {batch_id}")
print(f"  Table: {table_name} ({final_count} total messages)")
print(f"  HL7 files: {hl7_output_path}")
print(f"  TXT files: {txt_output_path}")
print(f"  S3 Location: {storage_location_str}/{relative_path}/")
print(f"  Performance: {write_time:.1f}s ({len(messages_list)/write_time:.1f} files/sec)")
print(f"{'='*70}")